# WORFLOW FROM AURA AND JUPYTER-NOTEBOOK


## Neo4j Aura → Upload CSV → Fetch → Plot (Plotly + NetworkX)

This notebook:
1) Connects to Neo4j Aura  within jupyter notebook
2) Reads the CSV and builds node/relationship tables  
3) Uploads them to Aura (batched)  
4) Fetches everything back from Aura  
5) Builds a NetworkX graph and plots with Plotly  
6) Exports a run folder (CSV + Cypher + HTML + optional PNG/SVG)


## 1.0 Connect jupyter-notebook with neo4j

In [6]:
from neo4j import GraphDatabase

URI = "neo4j+s://c977fb54.databases.neo4j.io"
USER = "neo4j"
PASSWORD = "1ikLQEHSZT5SDMiXs_GFxnLBGAyNMNjOXPOrX94JOMg"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("✅ Connected to database")



✅ Connected to database


## 2.0 Reads the CSV and builds node/relationship tables


In [8]:
import pandas as pd

CSV_PATH = "V3_Erfassungstabelle_Psychotraumatology_Graph_database_modelling_nodes_Datagraphs_WS2526_FILLED.xlsm - trauma_slides_ONE_concept_per_s (3).csv"
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

required = {
    "module_code","module_name","slide_page",
    "main_concept","phase_category","Competence_type_guess","granularity_guess"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV missing columns: {sorted(missing)}")

df = df.copy()
df["main_concept"] = df["main_concept"].astype(str).str.strip()
df = df[df["main_concept"].notna() & (df["main_concept"] != "") & (df["main_concept"].str.lower() != "concept")].copy()

df["slide_id"] = (
    df["module_code"].astype(str).str.strip()
    + "::slide_"
    + df["slide_page"].astype(int).astype(str)
)

def mode_or_blank(s: pd.Series) -> str:
    m = s.mode()
    return m.iloc[0] if len(m) else ""

modules = df[["module_code","module_name"]].drop_duplicates().rename(columns={"module_code":"id","module_name":"name"})
modules["label"] = "Module"

slides = df[["slide_id","module_code","module_name","slide_page"]].drop_duplicates().rename(columns={"slide_id":"id"})
slides["label"] = "Slide"
slides["name"] = slides["module_code"].astype(str) + " / slide " + slides["slide_page"].astype(int).astype(str)

concepts = (
    df.groupby("main_concept", as_index=False)
      .agg(
          frequency=("main_concept","size"),
          modules=("module_code", lambda x: int(x.nunique())),
          phase_mode=("phase_category", mode_or_blank),
          competence_mode=("Competence_type_guess", mode_or_blank),
          granularity_mode=("granularity_guess", mode_or_blank),
      )
      .rename(columns={"main_concept":"id"})
)
concepts["label"] = "Concept"
concepts["name"] = concepts["id"]

phases = pd.DataFrame({"id": sorted(df["phase_category"].fillna("").astype(str).str.strip().replace("", "Unknown").unique())})
phases["label"] = "Phase"
phases["name"] = phases["id"]

competences = pd.DataFrame({"id": sorted(df["Competence_type_guess"].fillna("").astype(str).str.strip().replace("", "Unknown").unique())})
competences["label"] = "Competence"
competences["name"] = competences["id"]

granularities = pd.DataFrame({"id": sorted(df["granularity_guess"].fillna("").astype(str).str.strip().replace("", "Unknown").unique())})
granularities["label"] = "Granularity"
granularities["name"] = granularities["id"]

nodes = pd.concat([
    modules[["id","label","name"]],
    slides[["id","label","name"]],
    concepts[["id","label","name","frequency","modules","phase_mode","competence_mode","granularity_mode"]],
    phases[["id","label","name"]],
    competences[["id","label","name"]],
    granularities[["id","label","name"]],
], ignore_index=True).fillna("")

rels = []

# Module -> Slide
for r in slides.itertuples(index=False):
    rels.append({"source": str(r.module_code), "target": str(r.id), "type": "HAS_SLIDE", "value": ""})

# Slide -> Concept
for r in df[["slide_id","main_concept"]].drop_duplicates().itertuples(index=False):
    rels.append({"source": str(r.slide_id), "target": str(r.main_concept), "type": "FOCUSES_ON", "value": ""})

# Concept -> Phase/Competence/Granularity
for r in concepts[["id","phase_mode"]].itertuples(index=False):
    v = (str(r.phase_mode).strip() or "Unknown")
    rels.append({"source": str(r.id), "target": v, "type": "HAS_PHASE", "value": v})

for r in concepts[["id","competence_mode"]].itertuples(index=False):
    v = (str(r.competence_mode).strip() or "Unknown")
    rels.append({"source": str(r.id), "target": v, "type": "HAS_COMPETENCE", "value": v})

for r in concepts[["id","granularity_mode"]].itertuples(index=False):
    v = (str(r.granularity_mode).strip() or "Unknown")
    rels.append({"source": str(r.id), "target": v, "type": "HAS_GRANULARITY", "value": v})

rels_df = pd.DataFrame(rels).drop_duplicates().reset_index(drop=True)

print("✅ Built:", len(nodes), "nodes and", len(rels_df), "relationships")
print("✅ Example module ids:", df["module_code"].dropna().astype(str).unique()[:10])


✅ Built: 568 nodes and 1332 relationships
✅ Example module ids: ['M1' 'M2' 'M3' 'M4' 'M5' 'M6' 'M7' 'M8' 'M9' 'M10']


## 2.1 Markdown cell — Create constraints

In [9]:
CONSTRAINTS = [
    "CREATE CONSTRAINT module_id_unique IF NOT EXISTS FOR (m:Module) REQUIRE m.id IS UNIQUE",
    "CREATE CONSTRAINT slide_id_unique  IF NOT EXISTS FOR (s:Slide)  REQUIRE s.id IS UNIQUE",
    "CREATE CONSTRAINT concept_id_unique IF NOT EXISTS FOR (c:Concept) REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT phase_id_unique IF NOT EXISTS FOR (p:Phase) REQUIRE p.id IS UNIQUE",
    "CREATE CONSTRAINT comp_id_unique IF NOT EXISTS FOR (k:Competence) REQUIRE k.id IS UNIQUE",
    "CREATE CONSTRAINT gran_id_unique IF NOT EXISTS FOR (g:Granularity) REQUIRE g.id IS UNIQUE",
]
with driver.session() as session:
    for q in CONSTRAINTS:
        session.run(q)
print("✅ Constraints ensured")


✅ Constraints ensured


In [10]:
def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


## 3.0 Uploads them to Aura (batched)

## 3.1 Upload nodes (batched, by label)

In [13]:
node_rows = nodes.fillna("").to_dict(orient="records")

by_label = {}
for r in node_rows:
    by_label.setdefault(r["label"], []).append(r)

{k: len(v) for k, v in by_label.items()}
Q_MODULE = """
UNWIND $rows AS row
MERGE (n:Module {id: row.id})
SET n.name = row.name
"""

Q_SLIDE = """
UNWIND $rows AS row
MERGE (n:Slide {id: row.id})
SET n.name = row.name
"""

Q_CONCEPT = """
UNWIND $rows AS row
MERGE (n:Concept {id: row.id})
SET n.name = row.name,
    n.frequency = toInteger(coalesce(row.frequency, 0)),
    n.modules   = toInteger(coalesce(row.modules, 0)),
    n.phase_mode       = coalesce(row.phase_mode, ""),
    n.competence_mode  = coalesce(row.competence_mode, ""),
    n.granularity_mode = coalesce(row.granularity_mode, "")
"""

Q_PHASE = """
UNWIND $rows AS row
MERGE (n:Phase {id: row.id})
SET n.name = row.name
"""

Q_COMP = """
UNWIND $rows AS row
MERGE (n:Competence {id: row.id})
SET n.name = row.name
"""

Q_GRAN = """
UNWIND $rows AS row
MERGE (n:Granularity {id: row.id})
SET n.name = row.name
"""
def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

BATCH = 2000

with driver.session() as session:
    for batch in chunked(by_label.get("Module", []), BATCH):
        session.run(Q_MODULE, rows=batch)
    for batch in chunked(by_label.get("Slide", []), BATCH):
        session.run(Q_SLIDE, rows=batch)
    for batch in chunked(by_label.get("Concept", []), BATCH):
        session.run(Q_CONCEPT, rows=batch)
    for batch in chunked(by_label.get("Phase", []), BATCH):
        session.run(Q_PHASE, rows=batch)
    for batch in chunked(by_label.get("Competence", []), BATCH):
        session.run(Q_COMP, rows=batch)
    for batch in chunked(by_label.get("Granularity", []), BATCH):
        session.run(Q_GRAN, rows=batch)

print("✅ Nodes uploaded (by label)")


✅ Nodes uploaded (by label)


## 3.2 Upload relationships

In [14]:
REL_QUERY = """
UNWIND $rows AS row
MATCH (a {id: row.source})
MATCH (b {id: row.target})
WITH a,b,row

FOREACH (_ IN CASE WHEN row.type = "HAS_SLIDE" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_SLIDE]->(b) SET r.value = coalesce(row.value,"")
)
FOREACH (_ IN CASE WHEN row.type = "FOCUSES_ON" THEN [1] ELSE [] END |
  MERGE (a)-[r:FOCUSES_ON]->(b) SET r.value = coalesce(row.value,"")
)
FOREACH (_ IN CASE WHEN row.type = "HAS_PHASE" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_PHASE]->(b) SET r.value = coalesce(row.value,"")
)
FOREACH (_ IN CASE WHEN row.type = "HAS_COMPETENCE" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_COMPETENCE]->(b) SET r.value = coalesce(row.value,"")
)
FOREACH (_ IN CASE WHEN row.type = "HAS_GRANULARITY" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_GRANULARITY]->(b) SET r.value = coalesce(row.value,"")
)
"""
rel_rows = rels_df.fillna("").to_dict(orient="records")

with driver.session() as session:
    for batch in chunked(rel_rows, 5000):
        session.run(REL_QUERY, rows=batch)

print("✅ Relationships uploaded:", len(rel_rows))


✅ Relationships uploaded: 1332


## 3.3 Sanity check counts

In [15]:
with driver.session() as session:
    print("Modules:", session.run("MATCH (m:Module) RETURN count(m) AS c").single()["c"])
    print("Slides:", session.run("MATCH (s:Slide) RETURN count(s) AS c").single()["c"])
    print("Concepts:", session.run("MATCH (c:Concept) RETURN count(c) AS c").single()["c"])
    print("Phases:", session.run("MATCH (p:Phase) RETURN count(p) AS c").single()["c"])


Modules: 10
Slides: 312
Concepts: 236
Phases: 4


## 4.0 Fetches everything back from Aura

In [16]:
# =========================================================
# ONE-CELL: Fetch ALL nodes+rels from Aura Neo4j, Plot interactively (Plotly),
# and export a "run folder" with:
#  - exports/nodes_fetched.csv
#  - exports/rels_fetched.csv
#  - exports/cypher_fetch_used.cypher   (all queries used)
#  - plots/graph.html                  (interactive)
#  - plots/graph.png + plots/graph.svg (if kaleido installed)
# =========================================================

import datetime as dt
from pathlib import Path
import math

import pandas as pd
import networkx as nx
import plotly.graph_objects as go



# ---------- Cypher queries (exported) ----------
FETCH_NODES_ALL = """
MATCH (n)
WHERE n.id IS NOT NULL
RETURN
  n.id AS id,
  labels(n)[0] AS label,
  coalesce(n.name, n.id) AS name,
  coalesce(n.frequency, 0) AS frequency,
  coalesce(n.modules, 0) AS modules,
  coalesce(n.phase_mode, "") AS phase_mode,
  coalesce(n.competence_mode, "") AS competence_mode,
  coalesce(n.granularity_mode, "") AS granularity_mode
"""

FETCH_EDGES_ALL = """
MATCH (a)-[r]->(b)
WHERE a.id IS NOT NULL AND b.id IS NOT NULL
RETURN
  a.id AS source,
  b.id AS target,
  type(r) AS type,
  coalesce(r.value, "") AS value
"""

(EXPORT_DIR / "cypher_fetch_used.cypher").write_text(
    "// QUERY 1: FETCH_NODES_ALL\n" + FETCH_NODES_ALL.strip()
    + "\n\n// QUERY 2: FETCH_EDGES_ALL\n" + FETCH_EDGES_ALL.strip()
    + "\n",
    encoding="utf-8"
)

# ---------- run queries ----------
with driver.session() as session:
    nodes_q = [dict(r) for r in session.run(FETCH_NODES_ALL)]
    edges_q = [dict(r) for r in session.run(FETCH_EDGES_ALL)]

nodes_df = pd.DataFrame(nodes_q)
edges_df = pd.DataFrame(edges_q)

# Save raw fetched data
nodes_df.to_csv(EXPORT_DIR / "nodes_fetched.csv", index=False, encoding="utf-8-sig")
edges_df.to_csv(EXPORT_DIR / "rels_fetched.csv", index=False, encoding="utf-8-sig")

print("✅ Fetched:", len(nodes_df), "nodes,", len(edges_df), "edges")
print("✅ Saved exports to:", EXPORT_DIR)



✅ Fetched: 568 nodes, 1332 edges
✅ Saved exports to: aura_runs_all_graph/run_20260210_180819/exports
✅ Saved interactive HTML: aura_runs_all_graph/run_20260210_180819/plots/graph.html
ℹ️ PNG/SVG not saved (install kaleido to enable): pip install -U kaleido
   Reason: ValueError('\nImage export using the "kaleido" engine requires the Kaleido package,\nwhich can be installed using pip:\n\n    $ pip install --upgrade kaleido\n')
✅ RUN_DIR: aura_runs_all_graph/run_20260210_180819


/home/caroline/.local/lib/python3.10/site-packages/plotly/io/_renderers.py:51: UserWarning:

Plotly version >= 6 requires Jupyter Notebook >= 7 but you have 6.4.8 installed.
 To upgrade Jupyter Notebook, please run `pip install notebook --upgrade`.



## 5.0 Builds a NetworkX graph and plots with Plotly

In [ ]:
# ---------- build graph ----------
G_multi = nx.MultiDiGraph()
for r in nodes_q:
    G_multi.add_node(str(r["id"]), **r)

for r in edges_q:
    G_multi.add_edge(str(r["source"]), str(r["target"]), key=r["type"], **r)

# layout on collapsed graph
G_simple = nx.Graph()
G_simple.add_nodes_from(G_multi.nodes(data=True))
for u, v, data in G_multi.edges(data=True):
    G_simple.add_edge(u, v)

pos = nx.spring_layout(
    G_simple,
    seed=42,
    k=0.6 / math.sqrt(max(1, G_simple.number_of_nodes()))
)

# ---------- colors ----------
NODETYPE_COLORS = {
    "Concept":     "#1F77B4",
    "Module":      "#2A9D8F",
    "Slide":       "#6C757D",
    "Phase":       "#D1495B",
    "Competence":  "#E9C46A",
    "Granularity": "#2CA02C",
}
NODE_FALLBACK = "#B0B0B0"

RELTYPE_COLORS = {
    "HAS_SLIDE": "#2F6FB0",
    "FOCUSES_ON": "#4B5563",
    "HAS_PHASE": "#1B6B7A",
    "HAS_COMPETENCE": "#4AC16D",
    "HAS_GRANULARITY": "#D1490F",
    "NEXT": "#0096A6",
    "CO_TAUGHT_WITH": "#8A8F98",
}
EDGE_NEUTRAL = "#B8BCC2"

# ---------- Plotly traces ----------
plotly_traces = []

# edges grouped by type
edge_types = sorted(set(edges_df["type"].tolist())) if len(edges_df) else []
for et in edge_types:
    edge_x, edge_y = [], []
    for u, v, k, data in G_multi.edges(keys=True, data=True):
        if data.get("type") != et:
            continue
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    plotly_traces.append(
        go.Scatter(
            x=edge_x, y=edge_y,
            mode="lines",
            line=dict(width=1.1, color=RELTYPE_COLORS.get(et, EDGE_NEUTRAL)),
            opacity=0.20,
            hoverinfo="none",
            name=et,
            showlegend=True
        )
    )

# nodes: one trace per label (nice legend)
labels_present = sorted({(d.get("label") or "Other") for _, d in G_simple.nodes(data=True)})
for lab in labels_present:
    xs, ys, hover = [], [], []
    for n, d in G_simple.nodes(data=True):
        if (d.get("label") or "Other") != lab:
            continue
        x, y = pos[n]
        xs.append(x); ys.append(y)
        hover.append(
            "<br>".join([
                f"<b>{n}</b>",
                f"type: {d.get('label','')}",
                f"name: {d.get('name','')}",
                f"frequency: {d.get('frequency','')}",
                f"modules: {d.get('modules','')}",
                f"phase_mode: {d.get('phase_mode','')}",
                f"competence_mode: {d.get('competence_mode','')}",
                f"granularity_mode: {d.get('granularity_mode','')}",
            ])
        )

    plotly_traces.append(
        go.Scatter(
            x=xs, y=ys,
            mode="markers",
            marker=dict(
                size=9,
                color=NODETYPE_COLORS.get(lab, NODE_FALLBACK),
                opacity=0.95,
                line=dict(width=0)
            ),
            hovertext=hover,
            hoverinfo="text",
            name=f"Node: {lab}",
            showlegend=True
        )
    )

fig = go.Figure(data=plotly_traces)
fig.update_layout(
    title="Aura Neo4j — Full Graph (interactive)",
    hovermode="closest",
    margin=dict(l=20, r=20, t=60, b=20),
    legend=dict(itemsizing="constant"),
)



## 6.0 Exports a run folder (CSV + Cypher + HTML + optional PNG/SVG)

In [ ]:
# ---------- output folders ----------
OUT_ROOT = Path("aura_runs_all_graph")
OUT_ROOT.mkdir(exist_ok=True)
RUN_DIR = OUT_ROOT / f"run_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXPORT_DIR = RUN_DIR / "exports"
PLOTS_DIR = RUN_DIR / "plots"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# Save interactive HTML
HTML_PATH = PLOTS_DIR / "graph.html"
fig.write_html(str(HTML_PATH), include_plotlyjs="cdn")
print("✅ Saved interactive HTML:", HTML_PATH)

# Try saving PNG/SVG if kaleido exists
PNG_PATH = PLOTS_DIR / "graph.png"
SVG_PATH = PLOTS_DIR / "graph.svg"
try:
    fig.write_image(str(PNG_PATH), width=1800, height=1200, scale=2)
    fig.write_image(str(SVG_PATH), width=1800, height=1200)
    print("✅ Saved PNG:", PNG_PATH)
    print("✅ Saved SVG:", SVG_PATH)
except Exception as e:
    print("ℹ️ PNG/SVG not saved (install kaleido to enable): pip install -U kaleido")
    print("   Reason:", repr(e))

print("✅ RUN_DIR:", RUN_DIR)

# Show interactively in notebook
fig.show()
